# 02. Data Preprocessing

This notebook selects the model features, identifies numerical and categorical columns, and prepares the preprocessing pipeline.

## 1. Load and clean data

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("../dataset/foodtimedataset.csv")

df.columns = df.columns.str.strip()

for col in df.select_dtypes(include="object").columns:
    df[col] = df[col].str.strip()

df["Delivery Time"] = pd.to_numeric(df["Delivery Time"], errors="coerce")
df = df.dropna(subset=["Delivery Time"])
df = df[df["Delivery Time"] > 0]

print("Cleaned dataset shape:", df.shape)
df.head()

Cleaned dataset shape: (9040, 17)


,ID,Delivery Person ID,Delivery Person Age,Delivery Person Ratings,Restaurant Latitude,Restaurant Longitude,Delivery Location Latitude,Delivery Location Longitude,Order Type,Vehicle Type,Temperature,Humidity,Precipitation,Weather Description,Traffic Level,Distance,Delivery Time
0,4607,INDORES13DEL02,37.0,4.9,22.745049,75.892471,22.765049,75.912471,Snack,motorcycle,17.11,77.0,0.0,haze,Low,NaN,21.666667
1,B379,BANGRES18DEL02,34.0,4.5,12.913041,77.683237,13.043041,77.813237,Snack,scooter,19.50,93.0,0.0,mist,Very High,37.17,85.266667
2,5D6D,BANGRES19DEL01,23.0,4.4,12.914264,77.678400,12.924264,77.688400,Drinks,motorcycle,20.45,91.0,0.0,mist,Low,3.34,28.583333
3,7A6A,COIMBRES13DEL02,38.0,4.7,11.003669,76.976494,11.053669,77.026494,Buffet,motorcycle,23.86,78.0,0.0,mist,Moderate,10.05,35.183333
4,70A2,CHENRES12DEL01,32.0,4.6,12.972793,80.249982,13.012793,80.289982,Snack,scooter,26.55,87.0,0.0,mist,High,9.89,43.450000


## 2. Select features and target

In [2]:
X = df.drop(columns=["ID", "Delivery Person ID", "Delivery Time"])
y = df["Delivery Time"]

print("Features shape:", X.shape)
print("Target shape:", y.shape)
X.head()

Features shape: (9040, 14)
Target shape: (9040,)


,Delivery Person Age,Delivery Person Ratings,Restaurant Latitude,Restaurant Longitude,Delivery Location Latitude,Delivery Location Longitude,Order Type,Vehicle Type,Temperature,Humidity,Precipitation,Weather Description,Traffic Level,Distance
0,37.0,4.9,22.745049,75.892471,22.765049,75.912471,Snack,motorcycle,17.11,77.0,0.0,haze,Low,NaN
1,34.0,4.5,12.913041,77.683237,13.043041,77.813237,Snack,scooter,19.50,93.0,0.0,mist,Very High,37.17
2,23.0,4.4,12.914264,77.678400,12.924264,77.688400,Drinks,motorcycle,20.45,91.0,0.0,mist,Low,3.34
3,38.0,4.7,11.003669,76.976494,11.053669,77.026494,Buffet,motorcycle,23.86,78.0,0.0,mist,Moderate,10.05
4,32.0,4.6,12.972793,80.249982,13.012793,80.289982,Snack,scooter,26.55,87.0,0.0,mist,High,9.89


## 3. Identify numerical and categorical columns

In [3]:
numeric_cols = X.select_dtypes(include=["number"]).columns.tolist()
categorical_cols = X.select_dtypes(exclude=["number"]).columns.tolist()

print("Numerical columns:", numeric_cols)
print("Categorical columns:", categorical_cols)

Numerical columns: ['Delivery Person Age', 'Delivery Person Ratings', 'Restaurant Latitude', 'Restaurant Longitude', 'Delivery Location Latitude', 'Delivery Location Longitude', 'Temperature', 'Humidity', 'Precipitation', 'Distance']
Categorical columns: ['Order Type', 'Vehicle Type', 'Weather Description', 'Traffic Level']


## 4. Create preprocessing pipelines

- Numerical missing values are filled with the median.
- Categorical missing values are filled with the most frequent value.
- One-Hot Encoding converts categories into numeric columns.

In [4]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_cols),
    ("categorical", categorical_pipeline, categorical_cols)
])

print("Preprocessing pipeline created successfully.")

Preprocessing pipeline created successfully.


## 5. Split into training and testing data

In [5]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

print("Training samples:", X_train.shape[0])
print("Testing samples:", X_test.shape[0])

Training samples: 7232
Testing samples: 1808
